In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings("ignore")

# ── Load CSVs ──────────────────────────────────────────
df_merged = pd.read_csv("../cleaned_data/merged_data.csv")
df_catalog = pd.read_csv("../raw_data/amazon_india_products_catalog.csv")

print(f"Merged data shape: {df_merged.shape}")
print(f"Catalog shape: {df_catalog.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'amazon_india_products_catalog.csv'

In [8]:
print(f"Shape: {df_catalog.shape}")
print(f"\nColumns: {df_catalog.columns.tolist()}")
print(f"\nNull counts:\n{df_catalog.isnull().sum()}")
print(f"\nSample:\n{df_catalog.head()}")

Shape: (2004, 11)

Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'base_price_2015', 'weight_kg', 'rating', 'is_prime_eligible', 'launch_year', 'model']

Null counts:
product_id           0
product_name         0
category             0
subcategory          0
brand                0
base_price_2015      0
weight_kg            0
rating               0
is_prime_eligible    0
launch_year          0
model                0
dtype: int64

Sample:
    product_id               product_name     category  subcategory  brand  \
0  PROD_000001  Apple iPhone 6 16GB Black  Electronics  Smartphones  Apple   
1  PROD_000002  Apple iPhone 6 32GB Black  Electronics  Smartphones  Apple   
2  PROD_000003  Apple iPhone 6 64GB Black  Electronics  Smartphones  Apple   
3  PROD_000004  Apple iPhone 6 16GB White  Electronics  Smartphones  Apple   
4  PROD_000005  Apple iPhone 6 32GB White  Electronics  Smartphones  Apple   

   base_price_2015  weight_kg  rating  is_prime_eligible  la

In [9]:
print(df_catalog.groupby("subcategory")["base_price_2015"].max().sort_values(ascending=False))

subcategory
Smartphones           323504.37
TV & Entertainment    296128.00
Laptops               199861.28
Tablets                79683.85
Smart Watch            49650.99
Audio                  24655.11
Name: base_price_2015, dtype: float64


In [10]:
# ── Fix Outliers in Catalog ────────────────────────────
subcategory_caps = {
    "Smart Watch":        50000,
    "Tablets":            80000,
    "Smartphones":        150000,
    "Laptops":            200000,
    "TV & Entertainment": 300000,
    "Audio":              50000,
}

for sub, cap in subcategory_caps.items():
    for pass_num in range(1, 4):
        mask = df_catalog["subcategory"] == sub
        prices = df_catalog.loc[mask, "base_price_2015"]

        Q1 = prices.quantile(0.25)
        Q3 = prices.quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = min(Q3 + 1.5 * IQR, cap)

        outlier_mask = mask & (df_catalog["base_price_2015"] > upper_bound)
        df_catalog.loc[outlier_mask, "base_price_2015"] = (
            df_catalog.loc[outlier_mask, "base_price_2015"] / 10
        ).round(2)

        print(f"{sub} (pass {pass_num}): {outlier_mask.sum()} outliers fixed (upper bound: ₹{upper_bound:,.2f})")

# Verify
print("\nAfter fix:")
print(df_catalog.groupby("subcategory")["base_price_2015"].max().sort_values(ascending=False))

Smart Watch (pass 1): 0 outliers fixed (upper bound: ₹50,000.00)
Smart Watch (pass 2): 0 outliers fixed (upper bound: ₹50,000.00)
Smart Watch (pass 3): 0 outliers fixed (upper bound: ₹50,000.00)
Tablets (pass 1): 0 outliers fixed (upper bound: ₹80,000.00)
Tablets (pass 2): 0 outliers fixed (upper bound: ₹80,000.00)
Tablets (pass 3): 0 outliers fixed (upper bound: ₹80,000.00)
Smartphones (pass 1): 258 outliers fixed (upper bound: ₹150,000.00)
Smartphones (pass 2): 83 outliers fixed (upper bound: ₹123,840.49)
Smartphones (pass 3): 182 outliers fixed (upper bound: ₹84,310.50)
Laptops (pass 1): 0 outliers fixed (upper bound: ₹200,000.00)
Laptops (pass 2): 0 outliers fixed (upper bound: ₹200,000.00)
Laptops (pass 3): 0 outliers fixed (upper bound: ₹200,000.00)
TV & Entertainment (pass 1): 0 outliers fixed (upper bound: ₹300,000.00)
TV & Entertainment (pass 2): 0 outliers fixed (upper bound: ₹300,000.00)
TV & Entertainment (pass 3): 0 outliers fixed (upper bound: ₹300,000.00)
Audio (pass 1):

In [11]:
print(df_catalog[df_catalog["subcategory"] == "Smartphones"]["base_price_2015"].describe())
print("\nTop 10 most expensive smartphones:")
print(df_catalog[df_catalog["subcategory"] == "Smartphones"]
      .nlargest(10, "base_price_2015")[["product_name", "base_price_2015"]])

count     1518.00000
mean     27763.27774
std      17239.28582
min       8432.09000
25%      15122.32500
50%      22189.25500
75%      36295.31500
max      84085.99000
Name: base_price_2015, dtype: float64

Top 10 most expensive smartphones:
                             product_name  base_price_2015
686     Samsung Galaxy Note 20 128GB Blue         84085.99
1118  Samsung Galaxy S23 Ultra 128GB Blue         83981.55
826        Samsung Galaxy S21 256GB White         83598.41
1425       OnePlus OnePlus 13 128GB White         83518.19
998    OnePlus OnePlus 10 Pro 256GB Black         83423.43
1413  Samsung Galaxy S25 Ultra 64GB White         83410.45
550         OnePlus OnePlus 7T 64GB White         83069.86
1424        OnePlus OnePlus 13 64GB White         83049.24
60           OnePlus OnePlus X 32GB White         82953.19
1297    OnePlus OnePlus Nord 4 128GB Blue         82683.90


In [18]:
# ── Check merged data ──────────────────────────────────
print("=== MERGED DATA ===")
print(f"Shape: {df_merged.shape}")
print(f"\nNull counts:\n{df_merged.isnull().sum()}")
print(f"\nDtypes:\n{df_merged.dtypes}")

# ── Check catalog ──────────────────────────────────────
print("\n=== CATALOG ===")
print(f"Shape: {df_catalog.shape}")
print(f"\nNull counts:\n{df_catalog.isnull().sum()}")
print(f"\nDtypes:\n{df_catalog.dtypes}")

=== MERGED DATA ===
Shape: (1116110, 35)

Null counts:
transaction_id                 0
order_date                     0
customer_id                    0
product_id                     0
product_name                   0
category                       0
subcategory                    0
brand                          0
original_price_inr             0
discount_percent               0
discounted_price_inr           0
quantity                       0
subtotal_inr                   0
final_amount_inr               0
customer_city                  0
customer_state                 0
customer_tier                  0
customer_spending_tier         0
customer_age_group             0
payment_method                 0
delivery_days                  0
delivery_type                  0
is_prime_member                0
is_festival_sale               0
festival_name                  0
customer_rating           338182
return_status                  0
order_month                    0
order_year           

In [21]:
df_merged.to_csv("cleaned_data/merged_data.csv", index=False)
df_catalog.to_csv("cleaned_data/products_catalog_cleaned.csv", index=False)
print(f"✅ Merged data saved: {len(df_merged):,} rows")
print(f"✅ Catalog saved: {len(df_catalog):,} rows")


✅ Merged data saved: 1,116,110 rows
✅ Catalog saved: 2,004 rows


In [23]:
df_merged = pd.read_csv("cleaned_data/merged_data.csv")
df_catalog = pd.read_csv("cleaned_data/products_catalog_cleaned.csv")

print("=== MERGED DATA ===")
print(f"Shape: {df_merged.shape}")
print(f"Remaining NaNs:\n{df_merged.isnull().sum()[df_merged.isnull().sum() > 0]}")

print("\n=== CATALOG ===")
print(f"Shape: {df_catalog.shape}")
print(f"Remaining NaNs: {df_catalog.isnull().sum().sum()}")

=== MERGED DATA ===
Shape: (1116110, 35)
Remaining NaNs:
customer_rating    338182
dtype: int64

=== CATALOG ===
Shape: (2004, 11)
Remaining NaNs: 0
